# sample_growth
3つの条件（Control / Treatment A / Treatment B）の OD600 の時間変化を、平均＋標準偏差で描きます。

**使い方：** このノートブックと同じフォルダに `growth.csv` を置き、上から順にすべてのセルを実行します。別の場所にあるデータを使うときは、最初の「設定」セルだけを書き換えてください。

In [ ]:
# 設定：データの場所と出力先（大学のPCで使うときは、ここだけ書き換える）
from pathlib import Path

# データのあるフォルダ（既定：このノートブックと同じフォルダ）
INPUT_DIR = Path(".")
# 読むデータファイルの名前
INPUT_FILE = "growth.csv"
# 図の保存先（既定：このノートブックと同じフォルダの中の sample_growth_output）
OUTPUT_DIR = INPUT_DIR / "sample_growth_output"

INPUT_PATH = INPUT_DIR / INPUT_FILE

In [ ]:
# 図の体裁の設定（仕様書 specs/sample_growth.md の値。通常は変更しない）
REQUIRED_COLUMNS = {"time_h", "condition", "replicate", "value"}
CONDITIONS = ("Control", "Treatment A", "Treatment B")
COLORS = ("#E69F00", "#0072B2", "#009E73")  # Okabe-Ito 配色
MARKERS = ("o", "s", "^")
FIGURE_SIZE = (8.5 / 2.54, 6.5 / 2.54)  # 幅 8.5 cm × 高さ 6.5 cm
FONT_SIZE = 8
DPI = 300
SD_DDOF = 1  # 標本標準偏差

In [ ]:
# ライブラリの読み込み
import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd

In [ ]:
# データの読み込みと検査（列・欠損値・条件名に問題があればエラーで止まる）
data = pd.read_csv(INPUT_PATH)
print(f"読み込んだファイル: {INPUT_PATH.resolve()}（{len(data)} 行）")
missing = REQUIRED_COLUMNS - set(data.columns)
if missing:
    raise ValueError(f"Missing required columns in {INPUT_PATH}: {', '.join(sorted(missing))}")
missing_values = sorted(column for column in REQUIRED_COLUMNS if data[column].isna().any())
if missing_values:
    raise ValueError(f"Missing values in required columns: {', '.join(missing_values)}")
unexpected = set(data["condition"]) - set(CONDITIONS)
if unexpected:
    raise ValueError(
        "Input conditions must be Control, Treatment A, or Treatment B. "
        f"Unexpected values: {sorted(unexpected)}"
    )

In [ ]:
# 集計：条件×時間ごとに、反復の平均と標準偏差を計算する（反復が2未満ならエラーで止まる）
grouped = data.groupby(["condition", "time_h"], sort=True)
replicate_counts = grouped.size()
undersized_groups = replicate_counts[replicate_counts < 2]
if not undersized_groups.empty:
    raise ValueError(
        "Each condition × time group requires at least 2 replicates. "
        f"Groups with fewer than 2 replicates: {undersized_groups.to_dict()}"
    )
summary = grouped["value"].agg(
    mean="mean", sd=lambda values: values.std(ddof=SD_DDOF)
)
summary

In [ ]:
# フォントと文字サイズの設定（Arial がなければ DejaVu Sans）
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
font_family = "Arial" if "Arial" in available_fonts else "DejaVu Sans"
plt.rcParams.update({
    "font.family": font_family,
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "pdf.fonttype": 42,
})
print(f"使用フォント: {font_family}")

In [ ]:
# 描画：平均の折れ線＋標準偏差のエラーバー（表示は最後のセルで行う）
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
for condition, color, marker in zip(CONDITIONS, COLORS, MARKERS):
    if condition not in summary.index.get_level_values("condition"):
        raise ValueError(f"No observations found for required condition: {condition}")
    points = summary.loc[condition].sort_index()
    ax.errorbar(
        points.index, points["mean"], yerr=points["sd"],
        label=condition, color=color, marker=marker,
        markersize=3, linewidth=1, elinewidth=0.8, capsize=2,
    )
ax.set_xlabel("Time (h)")
ax.set_ylabel("OD600")
ax.set_xlim(-1, 49)
ax.set_xticks([0, 12, 24, 36, 48])
ax.set_yscale("linear")
ax.margins(y=0.08)
ax.set_ylim(bottom=0)
ax.legend(loc="upper left", frameon=False)
fig.tight_layout(pad=1.08)
plt.close(fig)  # ここで自動表示されないようにする

In [ ]:
# 保存：PNG と PDF（bbox_inches="tight" は使わず、仕様の 8.5 × 6.5 cm を保つ）
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for extension in ("png", "pdf"):
    output_path = OUTPUT_DIR / f"sample_growth.{extension}"
    fig.savefig(
        output_path,
        dpi=DPI,
        metadata={"CreationDate": None, "ModDate": None} if extension == "pdf" else None,
    )
    print(f"保存しました: {output_path.resolve()}")

In [ ]:
# 図を画面に表示する
fig